In [1]:
!pip install medmnist

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# Install dependencies if needed
# !pip install medmnist scikit-image torch torchvision matplotlib tqdm

import numpy as np
import matplotlib.pyplot as plt
import time

import torch

from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

# Project modules
from dataset import NoisyDataset, get_dataloaders, NOISE_SIGMA, BATCH_SIZE
from model import ResU2Net
from losses import pinn_loss, laplacian, ssim_loss
from metrics import batch_psnr_ssim
from visualize import show_pairs, plot_history, show_test_results

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


## Dataset and DataLoader

We wrap BloodMNIST in a custom `NoisyDataset` that:
1. Loads the clean image (normalized to [0, 1]).
2. Adds **Gaussian noise** (σ = 25/255) on the fly to create the corrupted input.

This way the model always sees fresh noise samples, acting as light data augmentation.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    batch_size=BATCH_SIZE,
    sigma=NOISE_SIGMA,
    num_workers=2,
)


## Visualise DataLoader Samples

We display a batch of **clean** and corresponding **noisy** images to confirm the noise model looks reasonable.

In [ ]:
noisy_sample, clean_sample = next(iter(train_loader))
show_pairs(noisy_sample, clean_sample, n=8,
           title='BloodMNIST — Clean vs Noisy (σ=25/255)')


## Model Architecture — ResU2Net

In [ ]:
model = ResU2Net(in_ch=3, out_ch=3, base=16).to(DEVICE)

# Sanity check
dummy = torch.randn(2, 3, 28, 28).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
assert out.shape == dummy.shape, f'Shape mismatch: {out.shape}'

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input : {dummy.shape}  →  Output: {out.shape}')
print(f'Trainable parameters: {total_params:,}')


## Training

In [ ]:
NUM_EPOCHS   = 500
LR           = 1e-4
PATIENCE     = 50     # early stopping patience

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# History containers
history = {
    'train_loss': [], 'train_mse': [], 'train_pde': [],
    'train_ic':   [], 'train_bc':  [], 'train_ssim_loss': [], 'train_l1': [],
    'train_psnr': [], 'train_ssim': [],
    'val_loss':   [], 'val_mse':   [], 'val_pde':   [],
    'val_ic':     [], 'val_bc':    [], 'val_ssim_loss':   [], 'val_l1':   [],
    'val_psnr':   [], 'val_ssim':  [],
}

best_val_loss = float('inf')
epochs_no_improve = 0


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot = {k: 0.0 for k in ['loss','mse','pde','ic','bc','ssim','l1']}
    psnr_list, ssim_list = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for noisy, clean in loader:
            noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
            pred = model(noisy)
            loss, comps = pinn_loss(pred, clean, noisy)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            tot['loss'] += loss.item()
            for k, v in comps.items():
                tot[k] += v

            p, s = batch_psnr_ssim(pred, clean)
            psnr_list.append(p)
            ssim_list.append(s)

    n = len(loader)
    return {k: v / n for k, v in tot.items()} | {'psnr': np.mean(psnr_list), 'ssim': np.mean(ssim_list)}


print(f'Training for up to {NUM_EPOCHS} epochs (early stop patience={PATIENCE})\n')
print(f'{"Epoch":>6} | {"Tr Loss":>8} | {"Tr PSNR":>8} | {"Tr SSIM":>8} | {"Va Loss":>8} | {"Va PSNR":>8} | {"Va SSIM":>8}')
print('-' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)

    for k in ['loss','mse','pde','ic','bc','l1','psnr','ssim']:
        history[f'train_{k}'].append(tr[k])
        history[f'val_{k}'].append(va[k])
    history['train_ssim_loss'].append(tr['ssim_loss'] if 'ssim_loss' in tr else tr.get('ssim', 0))
    history['val_ssim_loss'].append(va['ssim_loss']   if 'ssim_loss' in va else va.get('ssim', 0))

    scheduler.step(va['loss'])

    print(f'{epoch:>6} | {tr["loss"]:>8.4f} | {tr["psnr"]:>8.2f} | {tr["ssim"]:>8.4f} '
          f'| {va["loss"]:>8.4f} | {va["psnr"]:>8.2f} | {va["ssim"]:>8.4f}')

    # Early stopping + checkpoint
    if va['loss'] < best_val_loss:
        best_val_loss = va['loss']
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}.')
            break

print('\nTraining complete. Loading best checkpoint.')
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

## Training History Plots

In [ ]:
plot_history(history)


## Test Set Evaluation

In [10]:
model.eval()

all_psnr_noisy, all_ssim_noisy = [], []
all_psnr_pred,  all_ssim_pred  = [], []
total_images = 0

# Store a few batches for visualisation
vis_noisy, vis_clean, vis_pred = [], [], []

t_start = time.time()

with torch.no_grad():
    for i, (noisy, clean) in enumerate(test_loader):
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
        pred = model(noisy)

        pn, sn = batch_psnr_ssim(noisy, clean)
        pp, sp = batch_psnr_ssim(pred,  clean)
        all_psnr_noisy.append(pn);  all_ssim_noisy.append(sn)
        all_psnr_pred.append(pp);   all_ssim_pred.append(sp)
        total_images += noisy.shape[0]

        if i < 2:   # save first 2 batches for display
            vis_noisy.append(noisy.cpu())
            vis_clean.append(clean.cpu())
            vis_pred.append(pred.cpu())

t_elapsed = time.time() - t_start
ms_per_img = (t_elapsed / total_images) * 1000

print('=' * 50)
print(f'  Test set results  ({total_images} images)')
print('=' * 50)
print(f'  Noisy input  →  PSNR: {np.mean(all_psnr_noisy):.2f} dB  |  SSIM: {np.mean(all_ssim_noisy):.4f}')
print(f'  Model output →  PSNR: {np.mean(all_psnr_pred):.2f} dB  |  SSIM: {np.mean(all_ssim_pred):.4f}')
print(f'  PSNR gain    →  Δ{np.mean(all_psnr_pred) - np.mean(all_psnr_noisy):.2f} dB')
print(f'  Inference time: {ms_per_img:.3f} ms / image  (total {t_elapsed:.2f}s)')
print('=' * 50)

  Test set results  (3421 images)
  Noisy input  →  PSNR: 20.67 dB  |  SSIM: 0.7081
  Model output →  PSNR: 29.46 dB  |  SSIM: 0.9418
  PSNR gain    →  Δ8.79 dB
  Inference time: 0.716 ms / image  (total 2.45s)


## Visual Results on the Test Set

In [ ]:
show_test_results(vis_noisy, vis_pred, vis_clean, n_show=10)